In [ ]:
# Install Fugu from source (skip if already installed)
# !pip install git+https://github.com/sandialabs/Fugu.git

In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from itertools import product

import fugu
from fugu import Scaffold, Brick
from fugu.bricks import Vector_Input
from fugu.backends import snn_Backend

In [ ]:
# ── Game definition ─────────────────────────────────────────────────────────
# Costs = years in prison (lower is better for each player).
# (Cooperate, Defect) means P1 cooperates, P2 defects → P1 gets 3 yrs, P2 gets 0.

ACTIONS = ['Cooperate', 'Defect']

payoffs = {
    ('Cooperate', 'Cooperate'): (1, 1),
    ('Cooperate', 'Defect'):    (3, 0),
    ('Defect',    'Cooperate'): (0, 3),
    ('Defect',    'Defect'):    (2, 2),
}

def find_nash_equilibria(actions, payoffs):
    """
    Pure strategy Nash equilibria: outcomes where no player can reduce
    their cost by unilaterally switching action.
    """
    nash = []
    for a1, a2 in product(actions, actions):
        c1, c2 = payoffs[(a1, a2)]
        p1_can_improve = any(payoffs[(alt, a2)][0] < c1 for alt in actions if alt != a1)
        p2_can_improve = any(payoffs[(a1, alt)][1] < c2 for alt in actions if alt != a2)
        if not p1_can_improve and not p2_can_improve:
            nash.append((a1, a2))
    return nash

true_nash = find_nash_equilibria(ACTIONS, payoffs)
print("Ground-truth Nash equilibria:", true_nash)

In [ ]:
# ── Fugu Brick ──────────────────────────────────────────────────────────────
#
# Architecture (Fugu paper, Section 4.3):
#   - Two source neurons (P1, P2), each driven by a Vector_Input at t=0
#   - For each action pair (a1, a2):
#       P1_source → pair neuron  with  delay = cost_p1(a1, a2)
#       P2_source → pair neuron  with  delay = cost_p2(a1, a2)
#   - Pair neuron: threshold ≈ 2, full leak each timestep (AND gate)
#     → fires only when both spikes arrive simultaneously,
#       i.e. only when cost_p1 == cost_p2

class NashEquilibrium_Brick(Brick):
    def __init__(self, actions, payoffs, name=None):
        super().__init__()
        self.name = name
        self.actions = actions
        self.payoffs = payoffs
        self.is_built = False
        self.supported_codings = fugu.input_coding_types
        self.metadata = {'D': max(c for pair in payoffs.values() for c in pair) + 1}

    def build(self, graph, metadata, controlled_nodes, inputs, input_codings):
        output_codings = [input_codings[0]]

        # Completion node (required by Fugu)
        completed_node = self.name + "_complete"
        graph.add_node(completed_node, index=-1, threshold=0.0, decay=0.0,
                       p=1.0, potential=0.0)
        graph.add_edge(controlled_nodes[0]['complete'], completed_node,
                       weight=1.0, delay=1.0)

        # Source neurons come in via inputs[0] (P1) and inputs[1] (P2)
        p1_source = inputs[0][0]
        p2_source = inputs[1][0]

        # One pair neuron per action combination
        output_nodes = []
        for idx, (a1, a2) in enumerate(product(self.actions, self.actions)):
            c1, c2 = self.payoffs[(a1, a2)]
            node_name = f"{self.name}_{a1}_{a2}"
            # threshold=1.9: fires when two weight-1 spikes arrive together (1+1=2>1.9)
            # decay=1.0: full leak, so a single early spike never accumulates
            graph.add_node(node_name, index=idx, threshold=1.9, decay=1.0,
                           p=1.0, potential=0.0)
            graph.add_edge(p1_source, node_name, weight=1.0, delay=float(c1))
            graph.add_edge(p2_source, node_name, weight=1.0, delay=float(c2))
            output_nodes.append(node_name)

        self.is_built = True
        return (graph, self.metadata, [{'complete': completed_node}],
                [output_nodes], output_codings)

In [ ]:
# ── Build scaffold & run ─────────────────────────────────────────────────────

scaffold = Scaffold()
scaffold.add_brick(Vector_Input(np.array([1]), coding="Raster", name="P1"), 'input')
scaffold.add_brick(Vector_Input(np.array([1]), coding="Raster", name="P2"), 'input')
scaffold.add_brick(
    NashEquilibrium_Brick(ACTIONS, payoffs, name='Nash'),
    [(0, 0), (1, 0)],
    output=True
)
scaffold.lay_bricks()
scaffold.summary(verbose=1)

backend = snn_Backend()
backend.compile(scaffold, {'record': 'all'})
result = backend.run(10)
print(result)

In [ ]:
# ── Parse results ────────────────────────────────────────────────────────────
#
# The snn_Backend records which neurons fired at which timestep.
# Pair neuron names encode the action pair, so we can read off the result.

fired_pairs = []
fire_times  = {}

for neuron_name, times in result.items():
    if neuron_name.startswith('Nash_') and neuron_name != 'Nash_complete':
        # name format: Nash_{a1}_{a2}
        _, a1, a2 = neuron_name.split('_', 2)
        if len(times) > 0:
            fired_pairs.append((a1, a2))
            fire_times[(a1, a2)] = min(times)

print("Pair neurons that fired (cost_p1 == cost_p2):")
for a1, a2 in fired_pairs:
    c1, c2 = payoffs[(a1, a2)]
    is_nash = (a1, a2) in true_nash
    print(f"  ({a1}, {a2})  costs=({c1},{c2})  t={fire_times[(a1,a2)]}  Nash={is_nash}")

print()
print("Ground-truth Nash:", true_nash)
print("Spiking fired:    ", fired_pairs)
print()
print("Note: pairs where cost_p1 == cost_p2 fire, which is a necessary but not")
print("sufficient condition for Nash equilibrium — see discussion below.")

## Discussion

The spiking network identifies action pairs where **cost_p1 == cost_p2** by exploiting
coincidence detection: a pair neuron fires only when both player spikes arrive at
the same timestep, which happens iff the two synaptic delays are equal.

For the Prisoner's Dilemma the costs are:

| | P2: C | P2: D |
|---|---|---|
| **P1: C** | (1, 1) | (3, 0) |
| **P1: D** | (0, 3) | (2, 2) |

- **(D, D)** fires at t=2 — this **is** the Nash equilibrium ✓  
- **(C, C)** fires at t=1 — this is a **spurious** fire; costs are equal (1,1) but
  P1 could defect for cost 0, so it is not Nash.

Equal costs are necessary but not sufficient for Nash equilibrium.
A post-processing filter (or a game where Nash pairs and equal-cost pairs coincide,
such as Stag Hunt below) gives a clean result.

In [ ]:
# ── Stag Hunt: a game where the spiking result is exact ──────────────────────
#
# Nash equilibria here are (Stag, Stag) and (Hare, Hare), both with equal costs.
# No spurious fires.

SH_ACTIONS = ['Stag', 'Hare']

# Costs = 4 - utility so lower is better.
# Utilities: (S,S)=(4,4), (S,H)=(0,3), (H,S)=(3,0), (H,H)=(3,3)
sh_payoffs = {
    ('Stag', 'Stag'): (0, 0),
    ('Stag', 'Hare'): (4, 1),
    ('Hare', 'Stag'): (1, 4),
    ('Hare', 'Hare'): (1, 1),
}

true_nash_sh = find_nash_equilibria(SH_ACTIONS, sh_payoffs)

scaffold_sh = Scaffold()
scaffold_sh.add_brick(Vector_Input(np.array([1]), coding="Raster", name="P1"), 'input')
scaffold_sh.add_brick(Vector_Input(np.array([1]), coding="Raster", name="P2"), 'input')
scaffold_sh.add_brick(
    NashEquilibrium_Brick(SH_ACTIONS, sh_payoffs, name='Nash'),
    [(0, 0), (1, 0)],
    output=True
)
scaffold_sh.lay_bricks()

backend_sh = snn_Backend()
backend_sh.compile(scaffold_sh, {'record': 'all'})
result_sh = backend_sh.run(10)

fired_sh = []
for neuron_name, times in result_sh.items():
    if neuron_name.startswith('Nash_') and neuron_name != 'Nash_complete':
        _, a1, a2 = neuron_name.split('_', 2)
        if len(times) > 0:
            fired_sh.append((a1, a2))

print("Stag Hunt")
print("Ground-truth Nash:", true_nash_sh)
print("Spiking fired:    ", fired_sh)
print("Perfect match:    ", set(fired_sh) == set(true_nash_sh))

In [ ]:
# ── Visualisation ────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Left: network diagram ────────────────────────────────────────────────────
G = nx.DiGraph()
G.add_node('P1'); G.add_node('P2')
pair_map = {f'({a1[0]},{a2[0]})': (a1, a2) for a1, a2 in product(ACTIONS, ACTIONS)}
for label in pair_map:
    G.add_node(label)

pos = {'P1': (0, 0.5), 'P2': (0, -0.5),
       '(C,C)': (2,  1.5), '(C,D)': (2,  0.5),
       '(D,C)': (2, -0.5), '(D,D)': (2, -1.5)}

for label, (a1, a2) in pair_map.items():
    c1, c2 = payoffs[(a1, a2)]
    G.add_edge('P1', label, delay=c1)
    G.add_edge('P2', label, delay=c2)

ax = axes[0]
nash_nodes     = [k for k, v in pair_map.items() if v in true_nash]
spurious_nodes = [k for k, v in pair_map.items() if v in fired_pairs and v not in true_nash]
silent_nodes   = [k for k, v in pair_map.items() if v not in fired_pairs]

nx.draw_networkx_nodes(G, pos, nodelist=['P1'],          node_color='#4C9BE8', node_size=2000, ax=ax)
nx.draw_networkx_nodes(G, pos, nodelist=['P2'],          node_color='#F28C38', node_size=2000, ax=ax)
nx.draw_networkx_nodes(G, pos, nodelist=nash_nodes,      node_color='#2ECC71', node_size=1800, ax=ax)
nx.draw_networkx_nodes(G, pos, nodelist=spurious_nodes,  node_color='#F1C40F', node_size=1800, ax=ax)
nx.draw_networkx_nodes(G, pos, nodelist=silent_nodes,    node_color='#BDC3C7', node_size=1800, ax=ax)

node_labels = {'P1': 'P1\nsource', 'P2': 'P2\nsource'}
for label, (a1, a2) in pair_map.items():
    c1, c2 = payoffs[(a1, a2)]
    t = fire_times.get((a1, a2), '—')
    node_labels[label] = f'{label}\n({c1},{c2})\nt={t}'

nx.draw_networkx_labels(G, pos, node_labels, font_size=8, font_weight='bold', ax=ax)
edge_labels = {(u, v): f"d={d['delay']}" for u, v, d in G.edges(data=True)}
nx.draw_networkx_edges(G, pos, arrows=True, arrowsize=15, edge_color='#777',
                       width=1.2, ax=ax, connectionstyle='arc3,rad=0.08')
nx.draw_networkx_edge_labels(G, pos, edge_labels, font_size=7, label_pos=0.35, ax=ax)

legend = [
    mpatches.Patch(color='#4C9BE8', label='P1 source (fires t=0)'),
    mpatches.Patch(color='#F28C38', label='P2 source (fires t=0)'),
    mpatches.Patch(color='#2ECC71', label='Nash equilibrium ✓'),
    mpatches.Patch(color='#F1C40F', label='Spurious fire (equal costs, not Nash)'),
    mpatches.Patch(color='#BDC3C7', label='Silent (asymmetric costs)'),
]
ax.legend(handles=legend, loc='upper center', fontsize=8, bbox_to_anchor=(0.5, -0.05))
ax.set_title("Fugu Spiking Network — Prisoner's Dilemma", fontsize=11)
ax.axis('off')

# ── Right: payoff matrix ─────────────────────────────────────────────────────
ax2 = axes[1]
matrix = np.array([[payoffs[(a1, a2)][0] for a2 in ACTIONS] for a1 in ACTIONS], dtype=float)
im = ax2.imshow(matrix, cmap='RdYlGn_r', vmin=0, vmax=3, aspect='auto')
plt.colorbar(im, ax=ax2, label='P1 cost (years)')
ax2.set_xticks([0, 1]); ax2.set_xticklabels(['P2: C', 'P2: D'], fontsize=11)
ax2.set_yticks([0, 1]); ax2.set_yticklabels(['P1: C', 'P1: D'], fontsize=11)
ax2.set_title("P1 Cost Matrix", fontsize=11)

for i, a1 in enumerate(ACTIONS):
    for j, a2 in enumerate(ACTIONS):
        c1, c2 = payoffs[(a1, a2)]
        marker = ' ★' if (a1, a2) in true_nash else (' ●' if (a1, a2) in fired_pairs else '')
        ax2.text(j, i, f'({c1},{c2}){marker}', ha='center', va='center',
                 fontsize=12, fontweight='bold',
                 color='white' if c1 >= 2.5 or c1 <= 0.5 else 'black')

ax2.text(0.5, -0.1, '★ = Nash eq.   ● = spurious spike',
         ha='center', transform=ax2.transAxes, fontsize=9, color='gray')

plt.tight_layout()
plt.savefig('fugu_nash_pd.png', dpi=150, bbox_inches='tight')
plt.show()